# NB01 — Data Engineering
**Global Oil Market Analysis**

Six datasets from three sources — World Bank, EIA, and FetchSeries Research. Each covers a different part of the market. This notebook loads, cleans, and merges them into a set of analysis-ready datasets that every other notebook draws from.

---

| Dataset | Source | Frequency | Coverage |
|---|---|---|---|
| Oil prices | World Bank Pink Sheets | Monthly | 1960-2026 |
| Production by country | EIA STEO | Monthly | 1993-2027 (incl. forecasts) |
| Consumption by region | EIA STEO | Monthly | 1990-2027 (incl. forecasts) |
| Global crude stocks | FetchSeries Research | Monthly | 2012-2025 |
| US crude stocks | EIA | Weekly | 1982-2026 |
| US tight oil (shale) | EIA STEO | Monthly | 2000-2026 |

---

## Sections
1. Load and parse all datasets
2. Clean — handle NA strings, standardize dates, resample US weekly stocks to monthly
3. Engineer key features
4. Build master datasets and export
5. Data quality summary

In [1]:
import pandas as pd
import numpy as np
import openpyxl
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 20)

# Update this path to point at your local data folder
DATA_DIR = './data/'
OUT_DIR  = './data/processed/'
os.makedirs(OUT_DIR, exist_ok=True)

print('Libraries loaded.')
print(f'Data directory : {DATA_DIR}')
print(f'Output directory: {OUT_DIR}')

Libraries loaded.
Data directory : ./data/
Output directory: ./data/processed/


## 1. Load and Parse All Datasets

Every file follows the same structure:
- Row 0: long column descriptions (skipped)
- Row 1: short column headers (used)
- Row 2+: data, with columns 0 and 1 being date string and datetime respectively

A helper function handles this pattern.

In [20]:
def load_excel_sheet(filepath, sheet_name, col_names=None):
    """
    Loads a sheet from the oil market Excel files.
    Row 0 = long descriptions (skip)
    Row 1 = short headers (use)
    Row 2+ = data
    Returns a clean DataFrame with 'date' as the index column.
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb[sheet_name]

    rows = []
    headers = None
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if i == 0:
            continue  # long descriptions, skip
        if i == 1:
            # Build headers: first two cols are date_str and date
            if col_names:
                headers = ['date_str', 'date'] + col_names
            else:
                headers = ['date_str', 'date'] + [str(c) for c in row[2:]]
            continue
        rows.append(list(row))

    wb.close()

    df = pd.DataFrame(rows, columns=headers)
    df['date'] = pd.to_datetime(df['date'])
    df = df.replace('NA', np.nan)
    df = df.drop(columns=['date_str'])
    df = df.dropna(subset=['date'])
    df = df.set_index('date')

    # Coerce all data columns to numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df

print('Loader function defined.')

Loader function defined.


In [19]:
# --- Prices ---
df_prices = load_excel_sheet(
    DATA_DIR + 'oil-prices-world-bank-pink-sheets.xlsx',
    'Oil prices',
    col_names=['price_world', 'price_europe', 'price_asia', 'price_north_america']
)
print(f'Prices: {df_prices.shape} | {df_prices.index.min().date()} to {df_prices.index.max().date()}')
df_prices.head(3)

Prices: (794, 4) | 1960-01-31 to 2026-02-28


,price_world,price_europe,price_asia,price_north_america
date,,,,
1960-01-31,1.63,1.63,1.63,NaN
1960-02-29,1.63,1.63,1.63,NaN
1960-03-31,1.63,1.63,1.63,NaN


In [23]:
# --- Production by country ---
df_prod = load_excel_sheet(
    DATA_DIR + 'oil-production-supply-by-country-eia-steo.xlsx',
    'Production by country'
)
print(f'Production: {df_prod.shape} | {df_prod.index.min().date()} to {df_prod.index.max().date()}')

# --- OPEC / OPEC+ production ---
df_opec_prod = load_excel_sheet(
    DATA_DIR + 'oil-production-supply-by-country-eia-steo.xlsx',
    'OPEC production',
    col_names=['production_opec', 'production_opec_plus']
)
print(f'OPEC prod:  {df_opec_prod.shape}  | {df_opec_prod.index.min().date()} to {df_opec_prod.index.max().date()}')

Production: (420, 44) | 1993-01-31 to 2027-12-31
OPEC prod:  (420, 2)  | 1993-01-31 to 2027-12-31


In [5]:
# --- Consumption by region ---
df_cons_region = load_excel_sheet(
    DATA_DIR + 'oil-consumption-demand-by-country-eia-steo.xlsx',
    'World',
    col_names=['cons_africa', 'cons_asia_oceania', 'cons_europe',
               'cons_former_soviet', 'cons_middle_east',
               'cons_north_america', 'cons_central_south_america', 'cons_world']
)
print(f'Consumption  : {df_cons_region.shape} | {df_cons_region.index.min().date()} to {df_cons_region.index.max().date()}')

# --- Consumption by country (major economies) ---
df_cons_country = load_excel_sheet(
    DATA_DIR + 'oil-consumption-demand-by-country-eia-steo.xlsx',
    'Countries',
    col_names=['cons_brazil', 'cons_canada', 'cons_china', 'cons_india',
               'cons_japan', 'cons_mexico', 'cons_russia', 'cons_usa']
)
print(f'Country cons : {df_cons_country.shape} | {df_cons_country.index.min().date()} to {df_cons_country.index.max().date()}')

Consumption  : (456, 8) | 1990-01-31 to 2027-12-31
Country cons : (456, 8) | 1990-01-31 to 2027-12-31


In [6]:
# --- Global crude stocks ---
df_global_stocks = load_excel_sheet(
    DATA_DIR + 'crude-oil-stocks-in-the-world-fsr.xlsx',
    'World oil stocks',
    col_names=['global_stocks_mmbbl']
)
print(f'Global stocks: {df_global_stocks.shape} | {df_global_stocks.index.min().date()} to {df_global_stocks.index.max().date()}')

Global stocks: (168, 1) | 2012-01-31 to 2025-12-31


In [7]:
# --- US crude stocks (weekly -> resample to monthly) ---
df_us_stocks_raw = load_excel_sheet(
    DATA_DIR + 'oil-stocks-in-the-us-eia.xlsx',
    'Oil stocks',
    col_names=['us_stocks_excl_spr', 'us_stocks_spr', 'us_stocks_total']
)
print(f'US stocks raw: {df_us_stocks_raw.shape} | weekly | {df_us_stocks_raw.index.min().date()} to {df_us_stocks_raw.index.max().date()}')

# Resample from weekly to monthly (last Friday of each month -> month-end mean)
df_us_stocks = df_us_stocks_raw.resample('ME').mean()
# Convert from thousand barrels to million barrels (same unit as global stocks)
df_us_stocks = df_us_stocks / 1000
df_us_stocks.columns = ['us_stocks_excl_spr_mmbbl', 'us_stocks_spr_mmbbl', 'us_stocks_total_mmbbl']
print(f'US stocks resampled: {df_us_stocks.shape} | monthly')

US stocks raw: (2267, 3) | weekly | 1982-08-20 to 2026-03-06
US stocks resampled: (524, 3) | monthly


In [8]:
# --- US tight oil / shale production ---
df_tight = load_excel_sheet(
    DATA_DIR + 'tight-oil-production-in-the-us-eia-steo.xlsx',
    'Tight oil production',
    col_names=['tight_total', 'tight_austin_chalk', 'tight_bakken',
               'tight_eagle_ford', 'tight_mississippian', 'tight_niobrara',
               'tight_permian', 'tight_woodford', 'tight_other']
)
print(f'Tight oil: {df_tight.shape} | {df_tight.index.min().date()} to {df_tight.index.max().date()}')

Tight oil: (314, 9) | 2000-01-31 to 2026-02-28


## 2. Data Quality Check

Check null rates and date coverage before merging.

In [9]:
# Null summary for each dataset
datasets = {
    'prices': df_prices,
    'production (country)': df_prod,
    'opec production': df_opec_prod,
    'consumption (region)': df_cons_region,
    'consumption (country)': df_cons_country,
    'global stocks': df_global_stocks,
    'us stocks (monthly)': df_us_stocks,
    'tight oil': df_tight,
}

quality = []
for name, df in datasets.items():
    total_cells = df.shape[0] * df.shape[1]
    null_cells  = df.isnull().sum().sum()
    quality.append({
        'dataset': name,
        'rows': df.shape[0],
        'cols': df.shape[1],
        'null_count': null_cells,
        'null_pct': round(null_cells / total_cells * 100, 1),
        'start': df.index.min().strftime('%Y-%m'),
        'end': df.index.max().strftime('%Y-%m'),
    })

pd.DataFrame(quality)

,dataset,rows,cols,null_count,null_pct,start,end
0,prices,794,4,264,8.30,1960-01,2026-02
1,production (country),420,44,286,1.50,1993-01,2027-12
2,opec production,420,2,0,0.00,1993-01,2027-12
3,consumption (region),456,8,24,0.70,1990-01,2027-12
4,consumption (country),456,8,24,0.70,1990-01,2027-12
5,global stocks,168,1,0,0.00,2012-01,2025-12
6,us stocks (monthly),524,3,0,0.00,1982-08,2026-03
7,tight oil,314,9,0,0.00,2000-01,2026-02


In [10]:
# Production: which countries have the most nulls?
# Some small producers have gaps early in the series
null_pct = (df_prod.isnull().sum() / len(df_prod) * 100).sort_values(ascending=False)
print('Production columns with >5% nulls:')
print(null_pct[null_pct > 5])

Production columns with >5% nulls:
Algeria                    5.24
Central African Republic   5.24
Venezuela                  5.24
United Arab Emirates       5.24
Iran                       5.24
Iraq                       5.24
Kuwait                     5.24
Libya                      5.24
Nigeria                    5.24
Gabon                      5.24
Equatorial Guinea          5.24
Congo                      5.24
Saudi Arabia               5.24
dtype: float64


## 3. Feature Engineering

Build the key derived columns that drive the analysis across all other notebooks.

In [11]:
# Pull US and world production into a cleaner working frame
df_prod_key = pd.DataFrame({
    'prod_usa':          df_prod['United States'],
    'prod_saudi':        df_prod['Saudi Arabia'],
    'prod_russia':       df_prod['Russia'],
    'prod_china':        df_prod['China'],
    'prod_iraq':         df_prod['Iraq'],
    'prod_iran':         df_prod['Iran'],
    'prod_canada':       df_prod['Canada'],
    'prod_uae':          df_prod['United Arab Emirates'],
    'prod_brazil':       df_prod['Brazil'],
    'prod_norway':       df_prod['Norway'],
    'prod_world':        df_prod['World'],
    'prod_opec':         df_opec_prod['production_opec'],
    'prod_opec_plus':    df_opec_prod['production_opec_plus'],
    'prod_tight_total':  df_tight['tight_total'],
    'prod_tight_permian':df_tight['tight_permian'],
    'prod_tight_bakken': df_tight['tight_bakken'],
    'prod_tight_eagle_ford': df_tight['tight_eagle_ford'],
})

# US shale as share of world production
df_prod_key['shale_share_of_world_pct'] = (
    df_prod_key['prod_tight_total'] / df_prod_key['prod_world'] * 100
)

# US production share of world
df_prod_key['us_share_of_world_pct'] = (
    df_prod_key['prod_usa'] / df_prod_key['prod_world'] * 100
)

# OPEC share of world
df_prod_key['opec_share_of_world_pct'] = (
    df_prod_key['prod_opec'] / df_prod_key['prod_world'] * 100
)

# Non-OPEC production
df_prod_key['prod_non_opec'] = df_prod_key['prod_world'] - df_prod_key['prod_opec']

print('Production features engineered.')
df_prod_key.tail(3)

Production features engineered.


,prod_usa,prod_saudi,prod_russia,prod_china,prod_iraq,prod_iran,prod_canada,prod_uae,prod_brazil,prod_norway,...,prod_opec,prod_opec_plus,prod_tight_total,prod_tight_permian,prod_tight_bakken,prod_tight_eagle_ford,shale_share_of_world_pct,us_share_of_world_pct,opec_share_of_world_pct,prod_non_opec
date,,,,,,,,,,,,,,,,,,,,,
2027-10-31,24.58,NaN,10.67,5.56,NaN,NaN,6.51,NaN,5.39,2.12,...,34.71,45.13,NaN,NaN,NaN,NaN,NaN,22.29,31.47,75.59
2027-11-30,24.76,NaN,10.71,5.58,NaN,NaN,6.64,NaN,5.28,2.15,...,34.65,45.07,NaN,NaN,NaN,NaN,NaN,22.39,31.34,75.92
2027-12-31,24.63,NaN,10.73,5.53,NaN,NaN,6.68,NaN,5.00,2.21,...,34.65,45.07,NaN,NaN,NaN,NaN,NaN,22.34,31.43,75.59


In [12]:
# Supply-demand balance: positive = surplus, negative = deficit
# Using world production and world consumption
df_balance = pd.DataFrame({
    'prod_world':  df_prod_key['prod_world'],
    'cons_world':  df_cons_region['cons_world'],
    'price_world': df_prices['price_world'],
})
df_balance['supply_demand_balance'] = df_balance['prod_world'] - df_balance['cons_world']
df_balance['balance_direction'] = df_balance['supply_demand_balance'].apply(
    lambda x: 'surplus' if x > 0 else ('deficit' if x < 0 else 'balanced')
)

# Price change metrics
df_balance['price_mom_pct'] = df_balance['price_world'].pct_change() * 100   # month-on-month
df_balance['price_yoy_pct'] = df_balance['price_world'].pct_change(12) * 100  # year-on-year
df_balance['price_rolling12'] = df_balance['price_world'].rolling(12).mean()  # 12-month moving avg

print('Supply-demand balance features engineered.')
df_balance.tail(3)

Supply-demand balance features engineered.


,prod_world,cons_world,price_world,supply_demand_balance,balance_direction,price_mom_pct,price_yoy_pct,price_rolling12
date,,,,,,,,
2027-10-31,110.30,106.23,NaN,4.07,surplus,0.00,0.00,NaN
2027-11-30,110.57,106.98,NaN,3.59,surplus,0.00,0.00,NaN
2027-12-31,110.25,108.21,NaN,2.04,surplus,0.00,0.00,NaN


In [13]:
# Price regime classification — historical context buckets
def classify_price_era(date):
    year = date.year
    if year < 1973:
        return '1. Pre-OPEC Era (<1973)'
    elif year < 1986:
        return '2. OPEC Shock Era (1973-1985)'
    elif year < 1999:
        return '3. Price Collapse Era (1986-1998)'
    elif year < 2008:
        return '4. China Demand Boom (1999-2007)'
    elif year < 2010:
        return '5. Financial Crisis (2008-2009)'
    elif year < 2015:
        return '6. High Price Era (2010-2014)'
    elif year < 2020:
        return '7. Shale Crash Era (2015-2019)'
    elif year < 2022:
        return '8. COVID Era (2020-2021)'
    elif year < 2024:
        return '9. Energy Crisis (2022-2023)'
    else:
        return '10. Current/Forecast (2024+)'

df_prices['price_era'] = df_prices.index.map(classify_price_era)
print('Price eras classified.')
df_prices.groupby('price_era')['price_world'].mean().round(2)

Price eras classified.


price_era
1. Pre-OPEC Era (<1973)              1.47
10. Current/Forecast (2024+)        72.52
2. OPEC Shock Era (1973-1985)       21.75
3. Price Collapse Era (1986-1998)   17.61
4. China Demand Boom (1999-2007)    39.00
5. Financial Crisis (2008-2009)     79.37
6. High Price Era (2010-2014)       97.67
7. Shale Crash Era (2015-2019)      55.22
8. COVID Era (2020-2021)            55.16
9. Energy Crisis (2022-2023)        88.93
Name: price_world, dtype: float64

## 4. Build Master Datasets and Export

Three output datasets:
- `master_full` — all 6 datasets merged, 2012-2025 overlap (all analyses that need stocks data)
- `master_supply_demand` — production + consumption + prices, 1993-2026 (broader supply/demand analysis)
- Individual cleaned datasets saved separately for notebooks that only need one source

In [14]:
# Full master — all 6 sources, 2012-2025 overlap
master_full = (
    df_prices
    .join(df_prod_key, how='inner')
    .join(df_cons_region, how='inner')
    .join(df_cons_country, how='inner')
    .join(df_global_stocks, how='inner')
    .join(df_us_stocks, how='inner')
    .join(df_balance[['supply_demand_balance', 'balance_direction',
                       'price_mom_pct', 'price_yoy_pct', 'price_rolling12']], how='inner')
)

# Trim to actuals only (not EIA forecasts) for master_full
master_full = master_full.loc[:'2025-12-31']

print(f'master_full: {master_full.shape}')
print(f'Date range : {master_full.index.min().date()} to {master_full.index.max().date()}')
master_full.head(3)

master_full: (168, 51)
Date range : 2012-01-31 to 2025-12-31


,price_world,price_europe,price_asia,price_north_america,price_era,prod_usa,prod_saudi,prod_russia,prod_china,prod_iraq,...,cons_usa,global_stocks_mmbbl,us_stocks_excl_spr_mmbbl,us_stocks_spr_mmbbl,us_stocks_total_mmbbl,supply_demand_balance,balance_direction,price_mom_pct,price_yoy_pct,price_rolling12
date,,,,,,,,,,,,,,,,,,,,,
2012-01-31,107.07,111.16,109.78,100.29,6. High Price Era (2010-2014),10.83,11.70,10.57,4.66,2.71,...,18.30,2224.21,309.30,695.95,1005.25,2.96,surplus,2.73,15.52,105.21
2012-02-29,112.69,119.70,116.15,102.21,6. High Price Era (2010-2014),10.94,11.90,10.56,4.62,2.61,...,18.62,2243.49,315.37,695.95,1011.32,-0.87,deficit,5.24,15.09,106.44
2012-03-31,117.78,124.93,122.28,106.15,6. High Price Era (2010-2014),10.95,11.89,10.56,4.65,2.76,...,18.13,2272.90,325.46,695.95,1021.41,0.82,surplus,4.52,8.41,107.20


In [15]:
# Broader supply-demand dataset — no stocks needed, goes back to 1993
master_supply_demand_full = (
    df_prod_key
    .join(df_cons_region, how='inner')
    .join(df_cons_country, how='inner')
    .join(df_prices[['price_world','price_era']], how='left')   # left = keep forecast months
    .join(df_balance[['supply_demand_balance','balance_direction',
                       'price_mom_pct','price_yoy_pct','price_rolling12']], how='left')
)
# Keep actuals up to Feb 2026, include EIA forecasts through 2027 separately
master_actuals  = master_supply_demand_full.loc[:'2026-02-28']
master_forecast = master_supply_demand_full.loc['2026-03-01':]

print(f'Actuals : {master_actuals.shape}')
print(f'Forecast: {master_forecast.shape}')

Actuals : (398, 44)
Forecast: (22, 44)


In [16]:
# Export all datasets as parquet (fast, preserves dtypes)
master_full.to_parquet(OUT_DIR + 'master_full.parquet')
master_actuals.to_parquet(OUT_DIR + 'master_actuals.parquet')
master_forecast.to_parquet(OUT_DIR + 'master_forecast.parquet')
df_prices.to_parquet(OUT_DIR + 'prices.parquet')
df_prod_key.to_parquet(OUT_DIR + 'production.parquet')
df_cons_region.to_parquet(OUT_DIR + 'consumption_region.parquet')
df_cons_country.to_parquet(OUT_DIR + 'consumption_country.parquet')
df_tight.to_parquet(OUT_DIR + 'tight_oil.parquet')
df_us_stocks.to_parquet(OUT_DIR + 'us_stocks.parquet')
df_global_stocks.to_parquet(OUT_DIR + 'global_stocks.parquet')

print('All datasets exported to data/processed/')

# Quick inventory
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(OUT_DIR + f)
    print(f'  {f:45s} {size:>8,} bytes')

All datasets exported to data/processed/
  consumption_country.parquet                     40,184 bytes
  consumption_region.parquet                      42,268 bytes
  global_stocks.parquet                            5,209 bytes
  master_actuals.parquet                         168,042 bytes
  master_forecast.parquet                         32,793 bytes
  master_full.parquet                            104,853 bytes
  prices.parquet                                  29,102 bytes
  production.parquet                              84,044 bytes
  tight_oil.parquet                               21,857 bytes
  us_stocks.parquet                               20,708 bytes


## 5. Data Quality Summary

In [17]:
print('=== MASTER_FULL ===')
print(f'Shape         : {master_full.shape}')
print(f'Date range    : {master_full.index.min().date()} to {master_full.index.max().date()}')
print(f'Null cells    : {master_full.isnull().sum().sum()}')
print(f'Null pct      : {master_full.isnull().sum().sum() / master_full.size * 100:.1f}%')

print()
print('=== MASTER_ACTUALS (supply-demand, broad) ===')
print(f'Shape         : {master_actuals.shape}')
print(f'Date range    : {master_actuals.index.min().date()} to {master_actuals.index.max().date()}')

print()
print('Key column nulls in master_full:')
key_cols = ['price_world', 'prod_world', 'cons_world', 'supply_demand_balance',
            'global_stocks_mmbbl', 'us_stocks_total_mmbbl', 'prod_tight_total']
print(master_full[key_cols].isnull().sum())

=== MASTER_FULL ===
Shape         : (168, 51)
Date range    : 2012-01-31 to 2025-12-31
Null cells    : 0
Null pct      : 0.0%

=== MASTER_ACTUALS (supply-demand, broad) ===
Shape         : (398, 44)
Date range    : 1993-01-31 to 2026-02-28

Key column nulls in master_full:
price_world              0
prod_world               0
cons_world               0
supply_demand_balance    0
global_stocks_mmbbl      0
us_stocks_total_mmbbl    0
prod_tight_total         0
dtype: int64


In [18]:
# Final snapshot — key metrics at the most recent actual date
latest = master_full.dropna(subset=['price_world', 'prod_world', 'cons_world']).iloc[-1]
print(f'Most recent actual data point: {latest.name.date()}')
print(f'  World oil price          : ${latest["price_world"]:.2f}/bbl')
print(f'  World production         : {latest["prod_world"]:.1f} Mb/d')
print(f'  World consumption        : {latest["cons_world"]:.1f} Mb/d')
print(f'  Supply-demand balance    : {latest["supply_demand_balance"]:+.2f} Mb/d ({latest["balance_direction"]})')
print(f'  US shale production      : {latest["prod_tight_total"]:.2f} Mb/d')
print(f'  US shale share of world  : {latest["shale_share_of_world_pct"]:.1f}%')
print(f'  OPEC share of world      : {latest["opec_share_of_world_pct"]:.1f}%')
print(f'  Global crude stocks      : {latest["global_stocks_mmbbl"]:.0f} million barrels')
print()
print('NB01 complete. Proceed to NB02 — Price History and Major Shocks.')

Most recent actual data point: 2025-12-31
  World oil price          : $60.88/bbl
  World production         : 108.0 Mb/d
  World consumption        : 106.0 Mb/d
  Supply-demand balance    : +2.06 Mb/d (surplus)
  US shale production      : 9.30 Mb/d
  US shale share of world  : 8.6%
  OPEC share of world      : 32.0%
  Global crude stocks      : 1848 million barrels

NB01 complete. Proceed to NB02 — Price History and Major Shocks.
